
STEP 1 — Create Colab Notebook

Go to:

Google Colab

Create:

linux_lfs_finetune.ipynb
STEP 2 — Enable GPU

In Colab:

Runtime
→ Change Runtime Type
→ GPU

Prefer:

T4
L4
A100 if available

T4 is enough for QLoRA.

STEP 3 — Install Dependencies

In [3]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl bitsandbytes

STEP 4 — Load Base Model

We’ll use:

Qwen2.5-Coder on Hugging Face

This prepares the model for efficient fine tuning.

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

STEP 5 — Add LoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Phase 2 — Dataset Creation

We’ll build:

Linux QA
LFS troubleshooting
shell debugging
build failures
boot errors

into structured JSONL.

STEP 6 — Create Dataset Folder

Inside Colab:

In [6]:
!mkdir datasets

STEP 7 — First Mini Dataset

Create:

In [9]:
import json

samples = [
    {
        "messages": [
            {
                "role": "user",
                "content": "Why does Linux From Scratch build a temporary toolchain?"
            },
            {
                "role": "assistant",
                "content": "The temporary toolchain isolates the host system and prevents contamination from host libraries and binaries during the final system build."
            }
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "What causes a kernel panic during boot?"
            },
            {
                "role": "assistant",
                "content": "Kernel panics during boot are often caused by missing initramfs files, filesystem mounting failures, incompatible kernel modules, or incorrect root filesystem parameters."
            }
        ]
    }
]

with open("datasets/linux_lfs_dataset.jsonl", "w") as f:
    for sample in samples:
        f.write(json.dumps(sample) + "\n")

Phase 2 — Build the Linux + LFS Dataset Pipeline

This is the most important part of the entire project.

A strong dataset beats a bigger model almost every time.

Goal

We want to transform:

Linux Docs
+
LFS Docs
+
Build Logs
+
Troubleshooting

into:

High Quality Instruction Examples
The Correct Dataset Architecture

Your dataset pipeline should become:

Raw Sources
    ↓
Cleaner
    ↓
Chunker
    ↓
Instruction Generator
    ↓
JSONL Formatter
    ↓
Training Dataset
PHASE 2.1 — Create Project Structure

In [15]:
!mkdir -p project/raw/html
!mkdir -p project/raw/logs
!mkdir -p project/processed
!mkdir -p project/final

STEP 2 — Install HTML Parsers


In [14]:
!pip install beautifulsoup4 lxml html5lib

STEP 4 — Parse HTML Properly


In [16]:
from bs4 import BeautifulSoup
from pathlib import Path

html_dir = Path("project/raw/html")

html_files = list(html_dir.rglob("*.html"))

html_docs = []

for html_file in html_files:

    with open(html_file, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f, "lxml")

    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()

    title = soup.title.text.strip() if soup.title else "Untitled"

    code_blocks = [
        pre.get_text("\n", strip=True)
        for pre in soup.find_all("pre")
    ]

    body = soup.get_text("\n", strip=True)

    html_docs.append({
        "source": str(html_file),
        "title": title,
        "body": body,
        "code_blocks": code_blocks,
    })

print(f"Loaded {len(html_docs)} HTML documents")

Loaded 2 HTML documents


STEP 3 — Parse Build Logs

This is EXTREMELY important.

In [17]:
log_dir = Path("project/raw/logs")

log_files = list(log_dir.rglob("*.log"))

logs = []

for log_file in log_files:

    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    logs.append({
        "source": str(log_file),
        "content": content
    })

print(f"Loaded {len(logs)} log files")

Loaded 9 log files


STEP 4 — Extract Errors from Logs

This becomes your troubleshooting intelligence.

In [18]:
import re

error_patterns = [
    r"error:.*",
    r"fatal:.*",
    r"undefined reference.*",
    r"No such file or directory.*",
    r"cannot find.*",
]

log_examples = []

for log in logs:

    lines = log["content"].splitlines()

    extracted = []

    for line in lines:

        for pattern in error_patterns:

            if re.search(pattern, line, re.IGNORECASE):
                extracted.append(line.strip())

    if extracted:

        log_examples.append({
            "source": log["source"],
            "errors": extracted[:20],
            "full_log": log["content"][:4000]
        })

print(f"Extracted {len(log_examples)} error examples")

Extracted 0 error examples


STEP 5 — Generate Linux QA Examples

Now we transform docs into training data.

In [19]:
dataset = []

for doc in html_docs:

    content = doc["body"][:2000]

    sample = {
        "messages": [
            {
                "role": "user",
                "content": f"Explain this Linux concept from {doc['title']}."
            },
            {
                "role": "assistant",
                "content": content
            }
        ]
    }

    dataset.append(sample)

print(len(dataset))

2


STEP 6 — Generate Troubleshooting Examples

THIS is the powerful part.

In [20]:
for log in log_examples:

    errors = "\n".join(log["errors"])

    sample = {
        "messages": [
            {
                "role": "user",
                "content": f"Help diagnose this Linux build failure:\n{errors}"
            },
            {
                "role": "assistant",
                "content": (
                    "This build failure likely relates to missing dependencies, "
                    "incorrect compiler configuration, broken include paths, "
                    "or toolchain contamination. Review the full build log and "
                    "verify required libraries and environment variables."
                )
            }
        ]
    }

    dataset.append(sample)

STEP 7 — Save Final Dataset

In [21]:
import json

output_path = "project/final/linux_dataset.jsonl"

with open(output_path, "w", encoding="utf-8") as f:
    for sample in dataset:
        f.write(json.dumps(sample) + "\n")

print(f"Saved {len(dataset)} training examples")

Saved 2 training examples


STEP 8 — Validate Dataset

VERY important.

Print a few examples

In [22]:
import random
import json

print(json.dumps(random.choice(dataset), indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "Explain this Linux concept from Beyond Linux\u00ae From Scratch (System V Edition)."
    },
    {
      "role": "assistant",
      "content": "Beyond Linux\u00ae From Scratch (System V Edition)\nBeyond\n              Linux\n\u00ae\nFrom Scratch\n(System\n              V\nEdition)\nVersion 12.4\nThe BLFS Development Team\nCopyright \u00a9 1999-2025 The BLFS Development Team\nCopyright \u00a9 1999-2025, The BLFS Development Team\nAll rights reserved.\nThis book is licensed under a\nCreative\n                Commons License\n.\nComputer instructions may be extracted from the book under\n                the\nMIT License\n.\nLinux\n\u00ae is a registered\n                trademark of Linus Torvalds.\nPublished 2025-09-01\nRevision History\nRevision 12.4\n2025-09-01\nThirty-second Release\nRevision 12.3\n2025-03-05\nThirty-first Release\nRevision 12.2\n2024-09-01\nThirtieth Release\nRevision 12.1\n2024-03-01\nTwenty-ninth Releas

Phase 3 — REAL QLoRA Fine Tuning

Now we train the model on your Linux + LFS dataset.

At this point you should already have:

project/final/linux_dataset.jsonl

Goal

We want to:

load dataset
tokenize chat format
fine tune with QLoRA
save adapters

NOT full model training.

STEP 1 — Install Remaining Dependencies

In [23]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 60.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.1 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.5 which 

STEP 2 — Load Dataset

In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="project/final/linux_dataset.jsonl",
    split="train"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 2
})


STEP 3 — Inspect Dataset

In [2]:
print(dataset[0])

{'messages': [{'role': 'user', 'content': 'Explain this Linux concept from Beyond Linux® From Scratch (System V Edition).'}, {'role': 'assistant', 'content': 'Beyond Linux® From Scratch (System V Edition)\nBeyond\n              Linux\n®\nFrom Scratch\n(System\n              V\nEdition)\nVersion 12.4\nThe BLFS Development Team\nCopyright © 1999-2025 The BLFS Development Team\nCopyright © 1999-2025, The BLFS Development Team\nAll rights reserved.\nThis book is licensed under a\nCreative\n                Commons License\n.\nComputer instructions may be extracted from the book under\n                the\nMIT License\n.\nLinux\n® is a registered\n                trademark of Linus Torvalds.\nPublished 2025-09-01\nRevision History\nRevision 12.4\n2025-09-01\nThirty-second Release\nRevision 12.3\n2025-03-05\nThirty-first Release\nRevision 12.2\n2024-09-01\nThirtieth Release\nRevision 12.1\n2024-03-01\nTwenty-ninth Release\nRevision 12.0\n2023-09-01\nTwenty-eighth Release\nRevision 11.3\n2023-

STEP 4 — Load Model Again

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.8.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

STEP 5 — Add LoRA

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

[transformers] Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


STEP 6 — Convert Dataset to Chat Template

VERY important.

Modern models expect formatted conversations.

In [5]:
def format_chat(example):

    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

Apply Formatting

In [6]:
dataset = dataset.map(format_chat)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Inspect Output

In [7]:
print(dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain this Linux concept from Beyond Linux® From Scratch (System V Edition).<|im_end|>
<|im_start|>assistant
Beyond Linux® From Scratch (System V Edition)
Beyond
              Linux
®
From Scratch
(System
              V
Edition)
Version 12.4
The BLFS Development Team
Copyright © 1999-2025 The BLFS Development Team
Copyright © 1999-2025, The BLFS Development Team
All rights reserved.
This book is licensed under a
Creative
                Commons License
.
Computer instructions may be extracted from the book under
                the
MIT License
.
Linux
® is a registered
                trademark of Linus Torvalds.
Published 2025-09-01
Revision History
Revision 12.4
2025-09-01
Thirty-second Release
Revision 12.3
2025-03-05
Thirty-first Release
Revision 12.2
2024-09-01
Thirtieth Release
Revision 12.1
2024-03-01
Twenty-ninth Release
Revision 12.0
2023-09-01
Twenty-eighth Re

STEP 7 — Configure Trainer

Now the actual fine tuning.

In [11]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        packing = True,
    ),
)

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


STEP 8 — Start Training

In [12]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.372889


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1/tokenizer_config.json.


TrainOutput(global_step=1, training_loss=1.3728891611099243, metrics={'train_runtime': 19.5507, 'train_samples_per_second': 0.102, 'train_steps_per_second': 0.051, 'total_flos': 61822941078528.0, 'train_loss': 1.3728891611099243})

STEP 9 — Save Adapters

VERY important.

In [13]:
model.save_pretrained("linux-lora")
tokenizer.save_pretrained("linux-lora")

[transformers] Unsloth: Restored added_tokens_decoder metadata in linux-lora/tokenizer_config.json.


('linux-lora/tokenizer_config.json',
 'linux-lora/chat_template.jinja',
 'linux-lora/tokenizer.json')

This saves ONLY LoRA adapters.

Not full gigantic model weights.

STEP 10 — Test Inference

Now test your Linux model.

Enable Inference

In [14]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

Generate Response

In [17]:
from transformers import TextStreamer

messages = [
    {
        "role": "user",
        "content": "Why does Linux From Scratch use a temporary toolchain?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True, # Added to prompt the assistant for a response
    return_tensors="pt"
).to("cuda")

# Provide attention_mask to avoid warnings and improve generation
attention_mask = (inputs != tokenizer.pad_token_id).long()

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    input_ids=inputs,
    attention_mask=attention_mask,
    max_new_tokens=256,
    temperature=0.7,
    streamer=text_streamer,
    use_cache=True
)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Why does Linux From Scratch use a temporary toolchain?<|im_end|>
<|im_start|>assistant


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Linux From Scratch (LFS) uses a temporary toolchain because it is necessary to build the basic system tools and libraries required to create a working Linux environment from scratch. This temporary toolchain is built using host utilities that are already available on the development machine, but it must be used carefully to ensure that the resulting binaries are compatible with the target architecture.

The temporary toolchain serves several important purposes:

1. **Building Basic Tools**: LFS starts by building essential tools like `make`, `gcc`, `glibc`, and other core utilities. These tools are needed to proceed with the construction of the rest of the system.

2. **Cross-Compilation**: The initial stage of LFS involves cross-compilation, where the host's compiler and tools are used to compile software for the target architecture. This is crucial because the final goal is to have a fully functional system running on the target hardware.

3. **Standardization**: By using a consisten

### STEP 11 — Save to Google Drive
Mount Google Drive to persist your LoRA adapters beyond this session.

In [19]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define the destination path in Google Drive
drive_path = '/content/drive/MyDrive/LFS_Finetune_Adapters'

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

# Copy the adapters from local storage to Google Drive
source_path = 'linux-lora'
if os.path.exists(source_path):
    # We use a loop or copytree; copytree requires the destination not to exist or use dirs_exist_ok
    shutil.copytree(source_path, drive_path, dirs_exist_ok=True)
    print(f"Successfully saved adapters to: {drive_path}")
else:
    print("Error: 'linux-lora' directory not found. Please run the saving cell first.")

Mounted at /content/drive
Successfully saved adapters to: /content/drive/MyDrive/LFS_Finetune_Adapters
